# RAG System Evaluation

This notebook evaluates the local Hugging Face text2text-generation pipeline through the running FastAPI service. No OpenAI API key is used. Run `evaluation/rag_evaluation.ipynb` for the canonical current evaluation workflow.

In [ ]:
import json
from pathlib import Path
import httpx

BASE_URL = 'http://127.0.0.1:8000'
ROOT = Path.cwd().parent if Path.cwd().name == 'evaluation' else Path.cwd()
QUESTIONS_PATH = ROOT / 'evaluation' / 'questions.json'
SAMPLE_PATH = ROOT / 'data' / 'sample_documents' / 'rag_fundamentals.txt'
questions = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))
health = httpx.get(f'{BASE_URL}/api/health', timeout=10)
health.raise_for_status()
print(f'API status: {health.json()["status"]}; evaluation questions: {len(questions)}')

## Prepare the sample source

The sample document is uploaded only if it is not already present in the live document library.

In [ ]:
documents_response = httpx.get(f'{BASE_URL}/api/documents', timeout=20)
documents_response.raise_for_status()
documents = documents_response.json()
if not any(item['filename'] == SAMPLE_PATH.name for item in documents):
    with SAMPLE_PATH.open('rb') as sample_file:
        upload = httpx.post(
            f'{BASE_URL}/api/documents/upload',
            files={'file': (SAMPLE_PATH.name, sample_file, 'text/plain')},
            timeout=120,
        )
    upload.raise_for_status()
    print(f"Indexed {upload.json()['filename']} with {upload.json()['chunk_count']} chunks")
else:
    print(f'{SAMPLE_PATH.name} is already indexed')

## Retrieval and answer review

For each question, the notebook inspects the actual API response and checks whether the expected document/page occurs in the returned sources. Review the displayed answer and enter one of the three labels when prompted.

In [ ]:
from IPython.display import Markdown, display

results = []
for number, item in enumerate(questions, start=1):
    response = httpx.post(
        f'{BASE_URL}/api/ask',
        json={'question': item['question']},
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    sources = payload['sources'][:5]
    expected_source = item.get('expected_source')
    expected_page = item.get('expected_page')
    retrieval_hit = any(
        source['filename'] == expected_source
        and (expected_page is None or source['page_number'] == expected_page)
        for source in sources
    )
    source_labels = [
        f"{source['filename']}:{source['page_number']} (score {source['score']:.2f})"
        for source in sources
    ]
    print(f'\n{number}. {item["question"]}')
    print(f'Expected: {item["expected_answer"]}')
    print(f'Generated: {payload["answer"]}')
    print(f'Retrieval hit@5: {retrieval_hit}; sources: {source_labels}')
    while True:
        classification = input('Answer classification (Correct / Partially Correct / Incorrect): ').strip().lower()
        labels = {'correct': 'Correct', 'partially correct': 'Partially Correct', 'incorrect': 'Incorrect'}
        if classification in labels:
            break
        print('Enter Correct, Partially Correct, or Incorrect.')
    results.append({
        'question': item['question'],
        'expected_answer': item['expected_answer'],
        'answer': payload['answer'],
        'retrieval_hit_at_5': retrieval_hit,
        'answer_classification': labels[classification],
        'sources': sources,
    })

total_questions = len(results)
if total_questions != len(questions):
    raise ValueError(f'Expected {len(questions)} evaluated questions, got {total_questions}.')
correct = sum(row['answer_classification'] == 'Correct' for row in results)
partially_correct = sum(row['answer_classification'] == 'Partially Correct' for row in results)
incorrect = sum(row['answer_classification'] == 'Incorrect' for row in results)
retrieval_hits = sum(row['retrieval_hit_at_5'] for row in results)
answer_accuracy = 100 * correct / total_questions
retrieval_accuracy = 100 * retrieval_hits / total_questions
print(f'\nAnswer accuracy: {answer_accuracy:.1f}% ({correct}/{total_questions} fully correct)')
print(f'Partially correct: {partially_correct}/{total_questions}; incorrect: {incorrect}/{total_questions}')
print(f'Retrieval accuracy / Recall@5: {retrieval_accuracy:.1f}% ({retrieval_hits}/{total_questions})')

def table_cell(value):
    return str(value).replace('|', '\\|').replace('\n', ' ')

summary_rows = [
    '| # | Question | Answer grade | Retrieval@5 | Retrieved sources |',
    '|---:|---|---|:---:|---|',
]
for number, result in enumerate(results, start=1):
    retrieved = '; '.join(
        f"{source['filename']} p.{source['page_number']} ({source['score']:.2f})"
        for source in result['sources']
    )
    summary_rows.append(
        f"| {number} | {table_cell(result['question'])} | {result['answer_classification']} "
        f"| {'Hit' if result['retrieval_hit_at_5'] else 'Miss'} | {table_cell(retrieved)} |"
    )
display(Markdown('\n'.join(summary_rows)))